In [76]:
import os
import numpy as np
import torch
import glob
import torch.nn as nn
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from torch.optim import Adam 
from torch.autograd import Variable
import torchvision
import pathlib

In [77]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [78]:
transformer = transforms.Compose([
    transforms.Resize((150,150)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),  #0-255 to 0-1
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) #0-1 t0 -1 -1
])

In [79]:
train = './data/seg_train/seg_train'
test = './data/seg_test/seg_test'

train_loder = DataLoader(
    torchvision.datasets.ImageFolder(train, transform=transformer),
    batch_size = 64,
    shuffle=True
)

test_loder = DataLoader(
    torchvision.datasets.ImageFolder(test, transform=transformer),
    batch_size = 64,
    shuffle=True
)

In [80]:
root = pathlib.Path(train)
classes = sorted([j.name.split('/')[-1] for j in root.iterdir()])
classes

['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

In [81]:

class ConvNet(nn.Module):
    def __init__(self, num_classes=6):
        super(ConvNet, self).__init__()
        
        self.conve1 = nn.Conv2d(in_channels=3, out_channels=12, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(num_features=12)
        self.relu1 = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2)
        
        self.conv2 = nn.Conv2d(in_channels=12, out_channels=20, kernel_size=3, stride=1, padding=1)
        self.relu2 =  nn.ReLU()
        
        
        self.conv3 = nn.Conv2d(in_channels=20, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(num_features=32)
        self.relu3 = nn.ReLU()
        
        self.fc = nn.Linear(in_features=32*75*75, out_features=num_classes)
        
    def forward(self, input):
        # print(input.shape)
        x = self.conve1(input)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool(x)
            
        x = self.conv2(x)
        x = self.relu2(x)
            
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        # print('x', x.shape)    
        x = x.view(-1, 32*75*75)
        x = self.fc(x)
            
        return x


In [82]:
model = ConvNet(num_classes=6).to(device)

In [83]:
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
loss_func = nn.CrossEntropyLoss()

In [84]:
num_epoches =10

In [85]:
train_count = len(glob.glob(train+'/**/*.jpg'))
test_count = len(glob.glob(test+'/**/*.jpg'))

In [86]:
print(train_count,test_count)

14034 3000


In [87]:
num_epoches =10

In [91]:
best_acc = 0
for epoch in range(num_epoches):
    print('start epoch : ', epoch)
    model.train()
    train_acc = 0
    train_loss = 0
    
    tr = len(glob.glob(train+'/**/*.jpg'))
    te = len(glob.glob(test+'/**/*.jpg'))
    
    for i , (images, labels) in enumerate(train_loder):
        print('iter : ', i, (tr - len(images)*(i+1)))
        if torch.cuda.is_available():
            images = Variable(images.cuda())
            labels = Variable(labels.cuda())
            
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = loss_func(outputs,labels)
        loss.backward()
        optimizer.step()
        
        train_loss+= loss.cpu().data*images.size(0)
        _,prediction = torch.max(outputs.data,1)
        
        train_acc += int(torch.sum(prediction==labels.data))
        
    train_acc = train_acc/train_count
    train_loss = train_loss/train_count

    model.eval()

    test_acc = 0
    for i, (images,labels) in enumerate(test_loder):
        print('te_iter : ', i, (test_count - len(images)*(i+1)))
        if torch.cuda.is_available():
            images=Variable(images.cuda())
            labels=Variable(labels.cuda())
            
        outputs=model(images)
        _,prediction=torch.max(outputs.data,1)
        test_acc += int(torch.sum(prediction==labels.data))
    
    test_acc = test_acc/test_count

    print('Epoch: '+str(epoch)+' Train Loss: '+str(train_loss)+' Train Accuracy: '+str(train_acc)+' Test Accuracy: '+str(test_acc)) 
    
    if test_acc>best_acc:
        torch.save(model.state_dict(),'best_checkpoint.model')
        best_acc=test_acc 
      
    

start epoch :  0


iter :  0 13970
iter :  1 13906
iter :  2 13842
iter :  3 13778
iter :  4 13714
iter :  5 13650
iter :  6 13586
iter :  7 13522
iter :  8 13458
iter :  9 13394
iter :  10 13330
iter :  11 13266
iter :  12 13202
iter :  13 13138
iter :  14 13074
iter :  15 13010
iter :  16 12946
iter :  17 12882
iter :  18 12818
iter :  19 12754
iter :  20 12690
iter :  21 12626
iter :  22 12562
iter :  23 12498
iter :  24 12434
iter :  25 12370
iter :  26 12306
iter :  27 12242
iter :  28 12178
iter :  29 12114
iter :  30 12050
iter :  31 11986
iter :  32 11922
iter :  33 11858
iter :  34 11794
iter :  35 11730
iter :  36 11666
iter :  37 11602
iter :  38 11538
iter :  39 11474
iter :  40 11410
iter :  41 11346
iter :  42 11282
iter :  43 11218
iter :  44 11154
iter :  45 11090
iter :  46 11026
iter :  47 10962
iter :  48 10898
iter :  49 10834
iter :  50 10770
iter :  51 10706
iter :  52 10642
iter :  53 10578
iter :  54 10514
iter :  55 10450
iter :  56 10386
iter :  57 10322
iter :  58 10258
iter : 

KeyboardInterrupt: 

In [ ]:
# import torch
# print(torch.cuda.is_available())  # Should print True if a GPU is available
# print(torch.cuda.device_count())  # Number of available GPUs
# print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# model = ConvNet(num_classes=6).to(device)



Using device: cpu
